# **Cotextual data extraction tool**
<font color="#FBB800">Agricultural Cold Chain Analysis and Prioritization (AgCAP) Tool</font><br>

-------
# Description
The main purpose of this notebook is to extract data from various contextual datasets to the settlements, sometimes through the voronoi-based influence areas, in order to add them to the SDI.<br>
It takes as input and enriches the processed extracted settlement data, generated with the **Core Analysis Tool** and stored in the `data/processed` directory, and performs the following operations:
1) Read the extracted settlement data (intermediate results)
2) Read the extracted voronoi outlines (intermediate results)
3) Read through the dataset in the foler 'raw/contextuals'
4) Extract the distance frome each selected contextuals data layer to the settlements
5) Extract the values from selected contextuals data layer to the settlements through the voronoi outlines
6) Merge all the data to the settlements layer, and export the updated settlement file to the 'data/processed/input_analyzed' folder for use in the AgCAP platform, or furhter preperation for the SDI upload

-------
# License and Copyright Notice

This notebook is part of the **AgCAP** project.

**Copyright (C) 2025 Sustainable Energy for All**

This program is free software: you can redistribute it and/or modify it under the terms of the **GNU Affero General Public License version 3 (AGPLv3)** as published by the Free Software Foundation.

This work is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU Affero General Public License for more details.

You should have received a copy of the GNU Affero General Public License along with this program. If not, see <https://www.gnu.org/licenses/agpl-3.0.html>.

**[Optional: Add a link to source code archive for AGPL compliance]**
*Download Source Code for this Version:* [Link to GitHub Tag Archive]

In [1]:
# Copyright (C) 2025 Sustainable Energy for All
#
#This program is free software: you can redistribute it and/or modify it 
# under the terms of the **GNU Affero General Public License version 3 (AGPLv3)** 
# as published by the Free Software Foundation.
#
# See <https://www.gnu.org/licenses/agpl-3.0.html>.

# Preparation

## Import packages and functions

In [2]:
import sys
from pathlib import Path

import numpy as np
import webbrowser
from threading import Timer

import gc
import warnings

import geopandas as gpd
import pandas as pd

import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

# Load the autoreload extension
%load_ext autoreload
%autoreload 2

current_dir = Path.cwd()
project_root = current_dir.parent

# 3. Insert the project root path into the system path
# This allows the notebook to find the 'scripts' folder as a package.
sys.path.insert(0, str(project_root))

# Import functions
from scripts.functions import *
from scripts.app import *

## Read input files

Make sure all the contextual layers that you wnat data to be extracted from sre in the 'raw/contextuals' data folder.

The voronoi outlien are extracted from the 'data/processed/input_voronoi' folder, and the settlements from teh latest output folder 'data/processed/input_analyzed/'

In [3]:
# --- Input: folder containing contextual layers (searched recursively) ---
CONTEXTUAL_DATA_DIR = Path(project_root/'data/raw/contextuals')

# --- Input: Voronoi boundaries ---
VORONOI_PATH = Path(project_root/'data/processed/input_voronoi/vor_poly_20260417.gpkg')

# --- Input: Settlements layer ---
SETTLEMENT_PATH = Path(project_root/'data/processed/input_analyzed/settlements_analyzed_20260423.gpkg')

# --- Output (timestamp appended to filename) ---
_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = SETTLEMENT_PATH.parent / f"{SETTLEMENT_PATH.stem}_contextuals_{_timestamp}.gpkg"

# --- Output (timestamp appended to filename) ---
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Output will be saved to: {OUTPUT_PATH}")

Output will be saved to: C:\Users\RobbertRaymondHoeboe\OneDrive - Sustainable Energy for All\Documents\Cooling\AgCAP\Mozambique\SDI_Data\AgCAP\data\processed\input_analyzed\settlements_analyzed_20260423_contextuals_20260716_113600.gpkg


## Select files for distance calculation

Select the files in the 'raw/contextuals' folder that you want to be included in the distance calcualtion

In [4]:
# List to hold the checkbox widgets
checkbox_widgets = []

# Output widget to display selected files
output_widget = widgets.Output()

# Global variable to store selected file paths
global selected_contextual_layers
selected_contextual_layers = []

print("Select the contextual layers you want to include in the analysis:")

# Iterate through files in CONTEXTUAL_DATA_DIR and create checkboxes
for filepath in CONTEXTUAL_DATA_DIR.rglob("*.gpkg"):
    if filepath.is_file():
        checkbox = widgets.Checkbox(
            value=False,
            description=filepath.name,
            disabled=False,
            indent=False
        )
        # Store the full filepath with the checkbox for easy retrieval
        checkbox.filepath = filepath
        checkbox_widgets.append(checkbox)

# Create a button to confirm selection
confirm_button = widgets.Button(description="Confirm Selection")

def on_confirm_button_click(b):
    global selected_contextual_layers
    selected_contextual_layers = []
    with output_widget:
        clear_output(wait=True)
        print("Selected files:")
        for cb in checkbox_widgets:
            if cb.value:
                selected_contextual_layers.append(str(cb.filepath))
                print(f"- {cb.description}")
        if not selected_contextual_layers:
            print("No files selected.")
        else:
            print(f"\nThe 'selected_contextual_layers' variable now contains {len(selected_contextual_layers)} file paths, ready for use in the next cell.")

confirm_button.on_click(on_confirm_button_click)

# Display the checkboxes and the button
display(widgets.VBox(checkbox_widgets), confirm_button, output_widget)


Select the contextual layers you want to include in the analysis:


Button(description='Confirm Selection', style=ButtonStyle())

Output()

## Distance calculation 

Distance is calcualted for each of the selected input files. Distance is calculated for each of the features in teh input files to the settlement. Results are stored in seperate columns. Calculations are done in the projects CRS 

In [12]:
# Load settlements layer
settlements_gdf = gpd.read_file(SETTLEMENT_PATH)

# Reset index to ensure uniqueness for later operations
settlements_gdf = settlements_gdf.reset_index(drop=True)

# Store original CRS
original_settlements_crs = settlements_gdf.crs

# Reproject settlements to a suitable projected CRS for accurate distance calculations.
TARGET_CRS = "EPSG:32736" # UTM Zone 36S for Mozambique

if original_settlements_crs.is_geographic or original_settlements_crs != TARGET_CRS:
    print(f"Reprojecting settlements from {original_settlements_crs} to {TARGET_CRS} for distance calculations.")
    settlements_gdf = settlements_gdf.to_crs(TARGET_CRS)
else:
    print(f"Settlements already in a projected CRS: {original_settlements_crs}. Using this for distance calculations.")

print(f"\nProcessing {len(selected_contextual_layers)} selected contextual layers.")
for filepath_str in selected_contextual_layers:
    filepath = Path(filepath_str)
    print(f"  Loading contextual layer: {filepath.name}")
    try:
        contextual_gdf = gpd.read_file(filepath)

        # Reset index for contextual_gdf to ensure uniqueness
        contextual_gdf = contextual_gdf.reset_index(drop=True)

        # Ensure contextual_gdf is in the same CRS as settlements_gdf (TARGET_CRS) for correct distance calculation
        if contextual_gdf.crs != settlements_gdf.crs:
            print(f"    Reprojecting '{filepath.name}' from {contextual_gdf.crs} to {settlements_gdf.crs}")
            contextual_gdf = contextual_gdf.to_crs(settlements_gdf.crs)

        # Perform spatial join nearest. This will add a 'temp_distance_m' column to the result,
        # containing the distance to the nearest feature in the contextual layer (in CRS units, typically meters).
        # Using a left join ensures all settlements are retained.
        # Set max_distance to 10,000 meters (10 km).
        temp_joined_gdf = gpd.sjoin_nearest(
            settlements_gdf, # Left GeoDataFrame (our settlements)
            contextual_gdf, # Right GeoDataFrame (the contextual layer)
            how="left", # Keep all rows from settlements_gdf
            max_distance=10000, # 10 km in meters
            distance_col='temp_distance_m' # Store calculated distance in a temporary column
        )

        # Generate a meaningful column name
        column_name = f"dist_to_{filepath.stem.lower()}_km" # Convert stem to lowercase for consistency

        # Check if the joined GeoDataFrame has duplicate indices
        if not temp_joined_gdf.index.is_unique:
            # If there are duplicate indices (meaning multiple nearest features for a single settlement),
            # group by the original settlement index and take the minimum distance.
            print(f"    Warning: Duplicate indices found in temp_joined_gdf for '{filepath.name}'. Taking minimum distance for each settlement.")
            min_distances_series = temp_joined_gdf.groupby(temp_joined_gdf.index)['temp_distance_m'].min()
            # Reindex this series to align with the original settlements_gdf index for assignment
            settlements_gdf[column_name] = min_distances_series.reindex(settlements_gdf.index) / 1000
        else:
            # If the index is unique, directly assign the distance, reindexing to ensure perfect alignment.
            settlements_gdf[column_name] = temp_joined_gdf['temp_distance_m'].reindex(settlements_gdf.index) / 1000

        print(f"    Added column: '{column_name}'")

    except Exception as e:
        print(f"    Error processing {filepath.name}: {e}")

# Display the first few rows of the updated settlements GeoDataFrame
print("\nUpdated Settlements GeoDataFrame with distances:")
display(settlements_gdf.head())

Reprojecting settlements from EPSG:4326 to EPSG:32736 for distance calculations.

Processing 3 selected contextual layers.
  Loading contextual layer: IDEPA_Aquaculture_Fisheries.gpkg
    Reprojecting 'IDEPA_Aquaculture_Fisheries.gpkg' from EPSG:4326 to EPSG:32736
    Added column: 'dist_to_idepa_aquaculture_fisheries_km'
  Loading contextual layer: IDEPA_Marine_Fisheries.gpkg
    Reprojecting 'IDEPA_Marine_Fisheries.gpkg' from EPSG:4326 to EPSG:32736
    Added column: 'dist_to_idepa_marine_fisheries_km'
  Loading contextual layer: MAAP_National_Ag_Areas.gpkg
    Reprojecting 'MAAP_National_Ag_Areas.gpkg' from EPSG:4326 to EPSG:32736
    Added column: 'dist_to_maap_national_ag_areas_km'

Updated Settlements GeoDataFrame with distances:


,id,Country,Province,District,Posto,Localidade,Urbanization Status,Cluster Area (from IEP),Cluster Area in km2,Population (from IEP),...,Fishing Activity (marine + inland),Fishing type,Fish Cooling Demand Export Market,Fish Cooling Demand National Market,Fish Cooling Demand Fresh Markets,Fish Cooling Demand ALL Markets,geometry,dist_to_idepa_aquaculture_fisheries_km,dist_to_idepa_marine_fisheries_km,dist_to_maap_national_ag_areas_km
0,4,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.521,0.445,88.13,...,NaN,none,NaN,NaN,NaN,NaN,"POLYGON ((342629.158 7513908.398, 342704.158 7...",NaN,NaN,5.630259
1,10,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.650,0.556,117.46,...,NaN,none,NaN,NaN,NaN,NaN,"POLYGON ((336348.753 7516251.181, 336498.753 7...",NaN,NaN,0.468197
2,13,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Rural Clusters,0.335,0.286,991.11,...,NaN,none,NaN,NaN,NaN,NaN,"POLYGON ((327500.441 7516531.171, 327500.441 7...",NaN,NaN,3.956056
3,14,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.329,0.282,60.94,...,NaN,none,NaN,NaN,NaN,NaN,"POLYGON ((330413.164 7516752.975, 330413.164 7...",NaN,NaN,4.123094
4,28,Mozambique,GAZA,CHICUALACUALA,VILA EDUARDO MONDLANE,Mbuzi,Low Density Rural,0.465,0.397,69.16,...,NaN,none,NaN,NaN,NaN,NaN,"POLYGON ((344547.888 7518218.311, 344472.888 7...",NaN,NaN,6.775326


## Reading files and attribute names - for value extraction

All the files in the 'raw/contextuals' folder are being shown, with the names of the attributes. This so the right dataset and attribute can be selected for extract values. In the Mozambique context, this is mainly for extracting fishing production information from teh IDEPA dataset

In [13]:
print(f"Files in '{CONTEXTUAL_DATA_DIR}':")
for filepath in CONTEXTUAL_DATA_DIR.rglob("*.gpkg"): # Filter for GeoPackage files
    if filepath.is_file():
        print(f"\n  Dataset: {filepath.name}")
        try:
            gdf = gpd.read_file(filepath)
            print(f"    Number of records: {len(gdf)}")
            print(f"    Attribute names: {list(gdf.columns)}")
            del gdf # Clear memory after use
            gc.collect()
        except Exception as e:
            print(f"    Error loading or processing {filepath.name}: {e}")

Files in 'C:\Users\RobbertRaymondHoeboe\OneDrive - Sustainable Energy for All\Documents\Cooling\AgCAP\Mozambique\SDI_Data\AgCAP\data\raw\contextuals':

  Dataset: IDEPA_All_Fisheries.gpkg
    Number of records: 723
    Attribute names: ['Distrito', 'Posto Administrativo', 'Centro de pesca', 'X', 'Y', 'Prod_Y_tn', 'Nr Pescadores', 'Nr Tanque', 'NrGaiolas', 'Tem enrgia(SIM;NÃO)', 'Type', 'geometry']

  Dataset: IDEPA_Aquaculture_Fisheries.gpkg
    Number of records: 294
    Attribute names: ['Provincia', 'Distrito', 'Aldeias', 'Y', 'X', 'Produção', 'Nr Aquacultores', 'Nr Tanque', 'NrGaiolas', 'Tem enrgia(SIM;NÃO)', 'geometry']

  Dataset: IDEPA_Marine_Fisheries.gpkg
    Number of records: 430
    Attribute names: ['Distrito', 'Posto Administrativo', 'Centro de pesca', 'X', 'Y', 'Produção/ano', 'Nr Pescadores', 'Tem enrgia(SIM;NÃO)', 'geometry']

  Dataset: MAAP_National_Ag_Areas.gpkg
    Number of records: 757
    Attribute names: ['ID', 'Nome das Areas', 'Distrito', 'Provincia', 'Area_P

## Select which file / attribute to extract!

In [14]:
# Define the filename for the contextual layer you want to use for value extraction (see datasets above)
DATAFILE_FOR_EXTRACTION = "IDEPA_All_Fisheries.gpkg" # <--- IMPORTANT: Change this to your desired file
ATTRIBUTE_TO_EXTRACT = "Prod_Y_tn" # <--- IMPORTANT: Change this to the desired attribute name from your selected layer

## Data values extraction

Value from the selected attribute will be extracted from the selected dataset, and attributed to the voronoi it is within. If mutliple values are present, the values are added up. Output will be the updated voronoi dataset with an additional column with the total sum of the values.

If multiple files need data value extraction, simply update the dataset above, and re-run the next cell!

In [15]:
LAYER_PATH_FOR_VALUE_EXTRACTION = CONTEXTUAL_DATA_DIR / DATAFILE_FOR_EXTRACTION

print(f"File for value extraction: {LAYER_PATH_FOR_VALUE_EXTRACTION.name}")
print(f"Attribute to extract: {ATTRIBUTE_TO_EXTRACT}")

# --- Load datasets ---
print(f"\nLoading contextual layer: {LAYER_PATH_FOR_VALUE_EXTRACTION.name}")
contextual_gdf = gpd.read_file(LAYER_PATH_FOR_VALUE_EXTRACTION)
print(f"Contextual layer CRS: {contextual_gdf.crs}")

print(f"Loading Voronoi boundaries: {VORONOI_PATH.name}")
voronoi_gdf = gpd.read_file(VORONOI_PATH)
print(f"Voronoi boundaries CRS: {voronoi_gdf.crs}")

# --- Ensure both GeoDataFrames are in the same projected CRS ---
# TARGET_CRS is defined in an earlier cell (s7ipgW5D63Xc)
if contextual_gdf.crs != TARGET_CRS:
    print(f"Reprojecting contextual layer from {contextual_gdf.crs} to {TARGET_CRS}")
    contextual_gdf = contextual_gdf.to_crs(TARGET_CRS)

if voronoi_gdf.crs != TARGET_CRS:
    print(f"Reprojecting Voronoi boundaries from {voronoi_gdf.crs} to {TARGET_CRS}")
    voronoi_gdf = voronoi_gdf.to_crs(TARGET_CRS)

print(f"CRS of contextual layer after reprojecting: {contextual_gdf.crs}")
print(f"CRS of Voronoi boundaries after reprojecting: {voronoi_gdf.crs}")

# --- Extract values using spatial join and summation ---
print(f"\nPerforming spatial join and summing '{ATTRIBUTE_TO_EXTRACT}' values...")

# Create a temporary unique ID for voronoi_gdf to ensure proper grouping after sjoin
voronoi_gdf['_original_idx_'] = voronoi_gdf.index

# Perform spatial join: 'left' join ensures all Voronoi polygons are retained
joined_gdf = gpd.sjoin(voronoi_gdf, contextual_gdf, how="left", predicate="intersects")

# Ensure the attribute column exists and is numeric for summing
if ATTRIBUTE_TO_EXTRACT not in joined_gdf.columns:
    print(f"Error: Attribute '{ATTRIBUTE_TO_EXTRACT}' not found in the joined GeoDataFrame. Please check the attribute name or selected layer.")
    # Add a placeholder column filled with 0 to voronoi_gdf
    voronoi_gdf[f'sum_{ATTRIBUTE_TO_EXTRACT}'] = 0
else:
    # Convert the attribute column to numeric, coercing errors (non-numeric values become NaN)
    # Then fill NaNs with 0 before summing to treat missing values as zero contribution.
    joined_gdf[ATTRIBUTE_TO_EXTRACT] = pd.to_numeric(joined_gdf[ATTRIBUTE_TO_EXTRACT], errors='coerce').fillna(0)

    # Group by the temporary original index of Voronoi polygons and sum the attribute
    summed_values = joined_gdf.groupby('_original_idx_')[ATTRIBUTE_TO_EXTRACT].sum().reset_index()
    summed_values = summed_values.rename(columns={ATTRIBUTE_TO_EXTRACT: f'sum_{ATTRIBUTE_TO_EXTRACT}'})

    # Merge the summed values back to the original Voronoi GeoDataFrame
    voronoi_gdf = voronoi_gdf.merge(summed_values, on='_original_idx_', how='left')

    # Fill NaN values in the new summed column (for Voronoi polygons that had no intersecting features)
    voronoi_gdf[f'sum_{ATTRIBUTE_TO_EXTRACT}'] = voronoi_gdf[f'sum_{ATTRIBUTE_TO_EXTRACT}'].fillna(0)

    print(f"Added column: 'sum_{ATTRIBUTE_TO_EXTRACT}' to Voronoi GeoDataFrame.")

# Drop the temporary index column
voronoi_gdf = voronoi_gdf.drop(columns=['_original_idx_'])

# Display the first few rows of the updated Voronoi GeoDataFrame
print("\nUpdated Voronoi GeoDataFrame with extracted values:")
display(voronoi_gdf.head())

# Clean up memory
del joined_gdf
del contextual_gdf
import gc
gc.collect()


File for value extraction: IDEPA_All_Fisheries.gpkg
Attribute to extract: Prod_Y_tn

Loading contextual layer: IDEPA_All_Fisheries.gpkg
Contextual layer CRS: EPSG:4326
Loading Voronoi boundaries: vor_poly_20260417.gpkg
Voronoi boundaries CRS: EPSG:4326
Reprojecting contextual layer from EPSG:4326 to EPSG:32736
Reprojecting Voronoi boundaries from EPSG:4326 to EPSG:32736
CRS of contextual layer after reprojecting: EPSG:32736
CRS of Voronoi boundaries after reprojecting: EPSG:32736

Performing spatial join and summing 'Prod_Y_tn' values...
Added column: 'sum_Prod_Y_tn' to Voronoi GeoDataFrame.

Updated Voronoi GeoDataFrame with extracted values:


,uid,uniqueID,Vor_area_sq.km,Vor_area_ha,geometry,sum_Prod_Y_tn
0,4.0,23548,220.013251,22001.325145,"MULTIPOLYGON (((334190.249 7483754.569, 331592...",0.0
1,10.0,23626,115.992775,11599.277500,"MULTIPOLYGON (((331888.149 7492386.03, 331592....",0.0
2,13.0,23686,76.005871,7600.587089,"MULTIPOLYGON (((328799.927 7512938.263, 329488...",0.0
3,14.0,23623,81.833827,8183.382656,"MULTIPOLYGON (((332332.426 7503003.348, 331259...",0.0
4,28.0,23676,53.625858,5362.585816,"MULTIPOLYGON (((339482.075 7519588.07, 338774....",0.0


0

## Extract results from Voronoi to Settlements dataset

Extracting the newly added total values column from the voronoi to the settlements dataset through the linkage of the ID/UID

In [16]:
# Select the relevant columns from voronoi_gdf for merging
# We need the unique identifier 'uid' and the new attribute (sum_Prod_Y_tn)
voronoi_attributes_to_merge = voronoi_gdf[['uid', f'sum_{ATTRIBUTE_TO_EXTRACT}']].copy()

# Perform a merge operation to add the attribute to the settlements_gdf
# Using 'id' from settlements_gdf and 'uid' from voronoi_gdf as the common identifiers
print(f"\nMerging attribute '{f'sum_{ATTRIBUTE_TO_EXTRACT}'}' from Voronoi data to settlements using settlement 'id' and Voronoi 'uid'...")
settlements_gdf = settlements_gdf.merge(
    voronoi_attributes_to_merge,
    left_on='id', # Column from settlements_gdf
    right_on='uid', # Column from voronoi_attributes_to_merge
    how='left' # Use a left merge to retain all settlements
)

# Drop the 'uid' column as it's no longer needed after the merge
settlements_gdf = settlements_gdf.drop(columns=['uid'])

# Fill NaN values for the newly merged column with 0,
# in case some settlements didn't match a Voronoi polygon (e.g., if they were outside the Voronoi extent)
settlements_gdf[f'sum_{ATTRIBUTE_TO_EXTRACT}'] = settlements_gdf[f'sum_{ATTRIBUTE_TO_EXTRACT}'].fillna(0)

print("\nSettlements GeoDataFrame after merging Voronoi attributes:")
display(settlements_gdf.head())
print(f"\nNew columns in settlements_gdf after merge: {list(settlements_gdf.columns)}")


Merging attribute 'sum_Prod_Y_tn' from Voronoi data to settlements using settlement 'id' and Voronoi 'uid'...

Settlements GeoDataFrame after merging Voronoi attributes:


,id,Country,Province,District,Posto,Localidade,Urbanization Status,Cluster Area (from IEP),Cluster Area in km2,Population (from IEP),...,Fishing type,Fish Cooling Demand Export Market,Fish Cooling Demand National Market,Fish Cooling Demand Fresh Markets,Fish Cooling Demand ALL Markets,geometry,dist_to_idepa_aquaculture_fisheries_km,dist_to_idepa_marine_fisheries_km,dist_to_maap_national_ag_areas_km,sum_Prod_Y_tn
0,4,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.521,0.445,88.13,...,none,NaN,NaN,NaN,NaN,"POLYGON ((342629.158 7513908.398, 342704.158 7...",NaN,NaN,5.630259,0.0
1,10,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.650,0.556,117.46,...,none,NaN,NaN,NaN,NaN,"POLYGON ((336348.753 7516251.181, 336498.753 7...",NaN,NaN,0.468197,0.0
2,13,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Rural Clusters,0.335,0.286,991.11,...,none,NaN,NaN,NaN,NaN,"POLYGON ((327500.441 7516531.171, 327500.441 7...",NaN,NaN,3.956056,0.0
3,14,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.329,0.282,60.94,...,none,NaN,NaN,NaN,NaN,"POLYGON ((330413.164 7516752.975, 330413.164 7...",NaN,NaN,4.123094,0.0
4,28,Mozambique,GAZA,CHICUALACUALA,VILA EDUARDO MONDLANE,Mbuzi,Low Density Rural,0.465,0.397,69.16,...,none,NaN,NaN,NaN,NaN,"POLYGON ((344547.888 7518218.311, 344472.888 7...",NaN,NaN,6.775326,0.0



New columns in settlements_gdf after merge: ['id', 'Country', 'Province', 'District', 'Posto', 'Localidade', 'Urbanization Status', 'Cluster Area (from IEP)', 'Cluster Area in km2', 'Population (from IEP)', 'Population 2025', 'Population density', 'Population within 20 km', 'Buildings (from IEP)', 'Buildings', 'Distance from MV power lines', 'Distance from nearest mini-grid', 'Nightlight (from IEP)', 'Night lights intensity', 'Electrification status nightlights', 'Electrification status (IEP study)', 'Least-cost electrification mode (IEP results)', 'Relative Wealth Index', 'Percentage of food-insecure people', 'Livelihoods', 'Conflict fatalities nearby', 'Cyclone hazard (1 in 100 years wind speed)', 'Elevation', 'Mean temperature', 'Mean diurnal temperature range', 'Cooling Degree Days Cold Room at 4C', 'Hot days (>30C) in a year', 'Relative humidity', 'Precipitation', 'Aridity Index', 'Climate class', 'Solar PV Output (kWh/year per installed kW)', 'Cropland area ha', 'Cropland percen

## Rounding values and renaming attributes - exporting result

Rounding the calcualteed distance values, and total extracted values to 3 decimal palces, and renaming them to more explicit column names

In [17]:
# Round decimals
settlements_gdf_rounded = settlements_gdf.round(3)

# Rename the columns before export
rename_dict={
 'dist_to_idepa_aquaculture_fisheries_km': 'Distance to IDEPA site (aquaculture)',
 'dist_to_idepa_marine_fisheries_km': 'Distance to IDEPA site (marine)',
 'dist_to_maap_national_ag_areas_km': 'Distance to MAAP site (irrigation)',
 'sum_Prod_Y_tn': 'Fish production - IDEPA (Tonnes/Year)'
 }

settlements_gdf_renamed = settlements_gdf_rounded.rename(columns=rename_dict)

display(settlements_gdf_renamed.head())

,id,Country,Province,District,Posto,Localidade,Urbanization Status,Cluster Area (from IEP),Cluster Area in km2,Population (from IEP),...,Fishing type,Fish Cooling Demand Export Market,Fish Cooling Demand National Market,Fish Cooling Demand Fresh Markets,Fish Cooling Demand ALL Markets,geometry,Distance to IDEPA site (aquaculture),Distance to IDEPA site (marine),Distance to MAAP site (irrigation),Fish production - IDEPA (Tonnes/Year)
0,4,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.521,0.445,88.13,...,none,NaN,NaN,NaN,NaN,"POLYGON ((342629.158 7513908.398, 342704.158 7...",NaN,NaN,5.630,0.0
1,10,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.650,0.556,117.46,...,none,NaN,NaN,NaN,NaN,"POLYGON ((336348.753 7516251.181, 336498.753 7...",NaN,NaN,0.468,0.0
2,13,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Rural Clusters,0.335,0.286,991.11,...,none,NaN,NaN,NaN,NaN,"POLYGON ((327500.441 7516531.171, 327500.441 7...",NaN,NaN,3.956,0.0
3,14,Mozambique,GAZA,CHICUALACUALA,PAFURI,Mbuzi,Low Density Rural,0.329,0.282,60.94,...,none,NaN,NaN,NaN,NaN,"POLYGON ((330413.164 7516752.975, 330413.164 7...",NaN,NaN,4.123,0.0
4,28,Mozambique,GAZA,CHICUALACUALA,VILA EDUARDO MONDLANE,Mbuzi,Low Density Rural,0.465,0.397,69.16,...,none,NaN,NaN,NaN,NaN,"POLYGON ((344547.888 7518218.311, 344472.888 7...",NaN,NaN,6.775,0.0


In [18]:
# Export the updated settlements_gdf to a GeoPackage file
print(f"\nExporting updated GeoDataFrame to: {OUTPUT_PATH}")
settlements_gdf.to_file(OUTPUT_PATH, driver="GPKG")
print("Export complete!")


Exporting updated GeoDataFrame to: C:\Users\RobbertRaymondHoeboe\OneDrive - Sustainable Energy for All\Documents\Cooling\AgCAP\Mozambique\SDI_Data\AgCAP\data\processed\input_analyzed\settlements_analyzed_20260423_contextuals_20260716_113600.gpkg
Export complete!
